# 02 · ETL demográfico y psicométrico

Construye las tablas centradas en el participante a partir de `Demographic and mental health data.csv`
y exporta dos JSON ligeros a `docs/data/`:

- **`participants.json`** — `DimParticipant`: id, sexo, BMI, altura, peso, grupo BMI, disponibilidad de sensores.
- **`psychometric_summary.json`** — subescalas SDQ y SNAP-IV por participante.

Los valores ausentes (8 participantes sin cuestionario, `age` y `SDQ19` inexistentes) se exportan como `null`.

In [1]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import etl_utils as eu
import pandas as pd

demo = pd.read_csv(os.path.join(eu.RAW_DIR, eu.DEMOGRAPHIC_FILE), encoding='utf-8')
inv = eu.list_sensor_files()
print('Participantes en demográfico:', len(demo), '| con ficheros de sensor:', len(inv))

Participantes en demográfico: 58 | con ficheros de sensor: 58


## 1. DimParticipant → `participants.json`

In [2]:
participants = []
for _, r in demo.iterrows():
    pid = str(r['ID'])
    files = inv.get(pid, {'F': None, 'T': None})
    participants.append({
        'participant_id': pid,
        'sex': (r['SEX'] if pd.notna(r['SEX']) else None),
        'age': None,  # ausente en el dataset pese a la descripción oficial
        'bmi': (float(r['BMI']) if pd.notna(r['BMI']) else None),
        'height_cm': (float(r['height(cm)']) if pd.notna(r['height(cm)']) else None),
        'weight_kg': (float(r['weight(kg)']) if pd.notna(r['weight(kg)']) else None),
        'bmi_group': eu.bmi_group(r['BMI']),
        'has_F': files['F'] is not None,
        'has_T': files['T'] is not None,
    })
print('Filas:', len(participants))
pd.DataFrame(participants).head()

Filas: 58


,participant_id,sex,age,bmi,height_cm,weight_kg,bmi_group,has_F,has_T
0,H1,female,None,15.4,125.0,24.0,normal,True,False
1,W4,male,None,15.9,128.0,26.0,normal,True,False
2,Z5,female,None,14.3,140.0,28.0,underweight,True,True
3,Z7,male,None,22.4,140.0,44.0,obesity,True,True
4,W53,female,None,19.2,135.0,35.0,overweight,False,True


## 2. Subescalas SDQ y SNAP-IV → `psychometric_summary.json`

In [3]:
psychometric = []
for _, r in demo.iterrows():
    sdq = eu.score_sdq(r)
    snap = eu.score_snap(r)
    rec = {'participant_id': str(r['ID'])}
    rec.update({f'sdq_{k}': v for k, v in sdq.items()})
    rec.update({f'snap_{k}': v for k, v in snap.items()})
    psychometric.append(rec)

psy_df = pd.DataFrame(psychometric)
print('Con SDQ válido:', int(psy_df['sdq_total_difficulties'].notna().sum()),
      '| sin cuestionario:', int(psy_df['sdq_emotional'].isna().sum()))
psy_df.describe().round(2)

Con SDQ válido: 50 | sin cuestionario: 8


,sdq_emotional,sdq_conduct,sdq_hyperactivity,sdq_peer,sdq_prosocial,sdq_total_difficulties,snap_inattention,snap_hyperactivity_impulsivity,snap_odd
count,50.00,50.00,50.00,50.00,50.00,50.00,49.00,49.00,49.00
mean,2.02,2.20,3.25,2.88,7.60,10.34,1.80,1.56,1.75
std,2.94,1.21,2.35,1.24,2.15,5.39,0.51,0.42,0.40
min,0.00,0.00,0.00,0.00,2.00,2.50,1.00,1.00,1.12
25%,0.00,1.00,1.00,2.50,6.00,6.50,1.44,1.22,1.50
50%,1.00,2.00,3.00,2.50,7.50,9.50,1.67,1.44,1.75
75%,3.00,3.00,5.00,3.75,10.00,11.94,2.11,1.67,1.88
max,18.00,6.00,9.00,6.25,10.00,29.50,3.11,2.56,3.00


## 3. Exportar JSON ligeros

In [4]:
print('Exportando a', eu.DATA_OUT_DIR)
eu.write_json(participants, 'participants.json')
eu.write_json(psychometric, 'psychometric_summary.json')
print('Hecho.')

Exportando a C:\Users\marco\Documents\18_Visualization_project_part_2\visualization_project_part_2\docs\data
  escrito participants.json  (8.2 KB)
  escrito psychometric_summary.json  (13.3 KB)
Hecho.
